In [ ]:
import pandas as pd

# Load data
covid = pd.read_csv("Covid Live.csv")

# Clean and rename columns
covid.columns = [
    'Index', 'Country', 'Total_Cases', 'Total_Deaths', 'New_Deaths',
    'Total_Recovered', 'Active_Cases', 'Serious_Critical',
    'Tot_Cases_per_Million', 'Deaths_per_Million', 'Total_Tests',
    'Tests_per_Million', 'Population'
]

In [ ]:
covid.dropna(subset=['Country'], inplace=True)
covid = covid[covid['Country'].str.strip() != '']

cols_to_clean = covid.columns.drop(['Index', 'Country'])
for col in cols_to_clean:
    covid[col] = covid[col].astype(str).str.replace(',', '').str.replace('+', '').str.strip()
    covid[col] = pd.to_numeric(covid[col], errors='coerce')

covid.dropna(thresh=10, inplace=True)
covid.drop(columns=['Index'], inplace=True)
covid.fillna(0, inplace=True)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Features and target
X = covid.drop(columns=['Country', 'Total_Deaths'])
y = covid['Total_Deaths']

# Normalize features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Neural network model
model = Sequential([
    Dense(64, input_dim=X_train.shape[1], activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X_train, y_train, epochs=100, batch_size=10, verbose=1)

# Evaluate
loss = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
